# Sesión 03 – ETL escalable con Spark

## Propósito
Construir un flujo ETL en Apache Spark aplicando transformaciones encadenadas, joins distribuidos y funciones ventana, validando la calidad de los datos y analizando decisiones iniciales de rendimiento.

## Caso
Se dispone de dos datasets:
- ventas: transacciones
- clientes: información de clientes

Se requiere integrar, transformar y analizar estos datos.

## Paso 1: Crear SparkSession

Inicializamos el entorno de trabajo en modo local.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion3-etl")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 01:55:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Extract – Leer datasets

Se cargan los archivos desde la carpeta `/opt/data`.
Para la sesión 3 se trabaja con dos datasets relacionados: `ventas.csv` y `clientes.csv`.

In [2]:
ventas = spark.read.csv("/opt/data/ventas.csv", header=True, inferSchema=True)
clientes = spark.read.csv("/opt/data/clientes.csv", header=True, inferSchema=True)

ventas.show()
clientes.show()

+---+----------+----------+-----+
| id|cliente_id|     fecha|monto|
+---+----------+----------+-----+
|  1|       101|2024-01-01|  100|
|  2|       101|2024-01-02|  200|
|  3|       102|2024-01-01|  150|
|  4|       103|2024-01-03|  300|
+---+----------+----------+-----+

+----------+------+
|cliente_id|nombre|
+----------+------+
|       101|  Juan|
|       102|   Ana|
|       103|  Luis|
+----------+------+



## 3. Explorar estructura

Antes de transformar, verificamos esquema (estructura) y tipos de datos.

In [3]:
ventas.printSchema()
clientes.printSchema()

root
 |-- id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- monto: integer (nullable = true)

root
 |-- cliente_id: integer (nullable = true)
 |-- nombre: string (nullable = true)



## 4. Transform – Limpieza básica

Aplicamos limpieza inicial:
- eliminar nulos críticos
- remover duplicados en clientes
- registros inconsistentes.

In [4]:
#ventas_limpio = ventas.dropna(subset=["id", "cliente_id", "fecha", "monto"])

from pyspark.sql.functions import col

ventas_limpio = (
    ventas
    .dropna(subset=["cliente_id", "fecha", "monto"]) # “Elimina registros donde falte cliente_id, fecha o monto”
    .filter(col("monto") > 0)
)

clientes_limpio = clientes.dropDuplicates(["cliente_id"])
ventas_limpio.show()
clientes_limpio.show()

+---+----------+----------+-----+
| id|cliente_id|     fecha|monto|
+---+----------+----------+-----+
|  1|       101|2024-01-01|  100|
|  2|       101|2024-01-02|  200|
|  3|       102|2024-01-01|  150|
|  4|       103|2024-01-03|  300|
+---+----------+----------+-----+

+----------+------+
|cliente_id|nombre|
+----------+------+
|       101|  Juan|
|       102|   Ana|
|       103|  Luis|
+----------+------+



🔎 **Pregunta:**
¿Por qué es importante limpiar datos antes de hacer un join?

## 5. Join distribuido

Integramos ventas con clientes mediante `cliente_id`. Este paso permite conectar información transaccional con atributos descriptivos.

In [5]:
df_join = ventas_limpio.join(clientes_limpio, "cliente_id", "inner")
df_join.show()

+----------+---+----------+-----+------+
|cliente_id| id|     fecha|monto|nombre|
+----------+---+----------+-----+------+
|       101|  2|2024-01-02|  200|  Juan|
|       101|  1|2024-01-01|  100|  Juan|
|       102|  3|2024-01-01|  150|   Ana|
|       103|  4|2024-01-03|  300|  Luis|
+----------+---+----------+-----+------+



🔎 **Pregunta:**
¿Qué ocurre internamente en Spark cuando se realiza este join?

(Shuffle = redistribución de datos entre nodos)
Spark mueve datos entre máquinas (o particiones) para poder agruparlos o combinarlos correctamente.

🔥 Problema:

Los datos del mismo cliente están en diferentes grupos (particiones)

💥 Solución (shuffle):

👉 Spark dice:

“Muevan todos los datos del cliente 101 al mismo lugar”

Alternativa optimizada :
from pyspark.sql.functions import broadcast

df = ventas_limpio.join(broadcast(clientes_limpio), "cliente_id")

## 6. Función ventana

Se calculan métricas por cliente:
- `ranking`: orden de compra por cliente
- `total_acumulado`: monto acumulado por cliente


In [6]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, sum

window_spec = Window.partitionBy("cliente_id").orderBy("fecha")

df_window = df_join.withColumn("ranking", row_number().over(window_spec))
df_window = df_join.withColumn("total_acumulado", sum("monto").over(window_spec))

#df_window = (
#    df_join
#    .withColumn("ranking", row_number().over(window_spec))
#    .withColumn("total_acumulado", sum("monto").over(window_spec))
#)

df_window.show()


+----------+---+----------+-----+------+---------------+
|cliente_id| id|     fecha|monto|nombre|total_acumulado|
+----------+---+----------+-----+------+---------------+
|       101|  1|2024-01-01|  100|  Juan|            100|
|       101|  2|2024-01-02|  200|  Juan|            300|
|       102|  3|2024-01-01|  150|   Ana|            150|
|       103|  4|2024-01-03|  300|  Luis|            300|
+----------+---+----------+-----+------+---------------+



📌 **Interpretación:**
- `partitionBy`: agrupa por cliente
- `orderBy`: ordena las compras
- `over`: aplica el cálculo sobre ese grupo

A diferencia de `groupBy`, no se pierden filas.

## 7. Validación de calidad de datos

Revisamos nulos por columna, cantidad de registros y posibles duplicados en el identificador de venta.

In [7]:
from pyspark.sql.functions import count, when

# Nulos
df_window.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_window.columns
]).show()

# Duplicados
df_window.groupBy("id").count().filter("count > 1").show()

# Conteo total
print("Total registros:", df_window.count())

+----------+---+-----+-----+------+---------------+
|cliente_id| id|fecha|monto|nombre|total_acumulado|
+----------+---+-----+-----+------+---------------+
|         0|  0|    0|    0|     0|              0|
+----------+---+-----+-----+------+---------------+

+---+-----+
| id|count|
+---+-----+
+---+-----+

Total registros: 4


In [8]:
print("Total de registros:", df_window.count())

df_window.groupBy("id").count().filter("count > 1").show()

Total de registros: 4
+---+-----+
| id|count|
+---+-----+
+---+-----+



## 8. Revisar decisiones iniciales de performance

Primero observamos el plan de ejecución del join y de la transformación con ventana.


In [9]:
df_window.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(total_acumulado, 'sum('monto) windowspecdefinition('cliente_id, 'fecha ASC NULLS FIRST, unspecifiedframe$()), None)]
+- Project [cliente_id#18, id#17, fecha#19, monto#20, nombre#39]
   +- Join Inner, (cliente_id#18 = cliente_id#38)
      :- Filter (monto#20 > 0)
      :  +- Filter atleastnnonnulls(3, cliente_id#18, fecha#19, monto#20)
      :     +- Relation [id#17,cliente_id#18,fecha#19,monto#20] csv
      +- Deduplicate [cliente_id#38]
         +- Relation [cliente_id#38,nombre#39] csv

== Analyzed Logical Plan ==
cliente_id: int, id: int, fecha: date, monto: int, nombre: string, total_acumulado: bigint
Project [cliente_id#18, id#17, fecha#19, monto#20, nombre#39, total_acumulado#142L]
+- Project [cliente_id#18, id#17, fecha#19, monto#20, nombre#39, total_acumulado#142L, total_acumulado#142L]
   +- Window [sum(monto#20) windowspecdefinition(cliente_id#18, fecha#19 ASC NULLS FIRST, specifiedwindowframe(RangeFrame, unbounded

In [10]:
from pyspark.sql.functions import broadcast

df_broadcast = ventas_limpio.join(broadcast(clientes_limpio), "cliente_id")
df_broadcast.explain(True)


== Parsed Logical Plan ==
'Join UsingJoin(Inner, [cliente_id])
:- Filter (monto#20 > 0)
:  +- Filter atleastnnonnulls(3, cliente_id#18, fecha#19, monto#20)
:     +- Relation [id#17,cliente_id#18,fecha#19,monto#20] csv
+- ResolvedHint (strategy=broadcast)
   +- Deduplicate [cliente_id#38]
      +- Relation [cliente_id#38,nombre#39] csv

== Analyzed Logical Plan ==
cliente_id: int, id: int, fecha: date, monto: int, nombre: string
Project [cliente_id#18, id#17, fecha#19, monto#20, nombre#39]
+- Join Inner, (cliente_id#18 = cliente_id#38)
   :- Filter (monto#20 > 0)
   :  +- Filter atleastnnonnulls(3, cliente_id#18, fecha#19, monto#20)
   :     +- Relation [id#17,cliente_id#18,fecha#19,monto#20] csv
   +- ResolvedHint (strategy=broadcast)
      +- Deduplicate [cliente_id#38]
         +- Relation [cliente_id#38,nombre#39] csv

== Optimized Logical Plan ==
Project [cliente_id#18, id#17, fecha#19, monto#20, nombre#359]
+- Join Inner, (cliente_id#18 = cliente_id#38), rightHint=(strategy=broadc

In [11]:
print("Particiones ventas:", ventas_limpio.rdd.getNumPartitions())
print("Particiones df final:", df_window.rdd.getNumPartitions())

Particiones ventas: 1
Particiones df final: 1


🔎 **Pregunta:**
¿Qué operaciones generan mayor costo en el plan de ejecución?

## 9. Load – Escribir salida en Parquet

La salida del ETL se guarda en formato Parquet dentro de `/opt/artifacts/output`.

In [12]:
df_window.write.mode("overwrite").parquet("/opt/artifacts/output")

## 10. Verificación de salida

Leemos nuevamente el resultado para comprobar que el ETL se generó correctamente.

In [13]:
salida = spark.read.parquet("/opt/artifacts/output")
salida.show()

+----------+---+----------+-----+------+---------------+
|cliente_id| id|     fecha|monto|nombre|total_acumulado|
+----------+---+----------+-----+------+---------------+
|       101|  1|2024-01-01|  100|  Juan|            100|
|       101|  2|2024-01-02|  200|  Juan|            300|
|       102|  3|2024-01-01|  150|   Ana|            150|
|       103|  4|2024-01-03|  300|  Luis|            300|
+----------+---+----------+-----+------+---------------+



## 11. Reto de extensión

1. Filtrar solo ventas mayores a 150 antes del join.
2. Agregar una columna de categoría de monto: bajo, medio, alto.
3. Justificar si conviene pensar en broadcast cuando una tabla es muy pequeña.

In [14]:
df_reto = (
    ventas_limpio
    .filter(col("monto") > 150)
    .join(clientes_limpio, "cliente_id", "inner")
    .withColumn(
        "categoria_monto",
        when(col("monto") < 150, "bajo")
        .when(col("monto") <= 250, "medio")
        .otherwise("alto")
    )
)

df_reto.show()

+----------+---+----------+-----+------+---------------+
|cliente_id| id|     fecha|monto|nombre|categoria_monto|
+----------+---+----------+-----+------+---------------+
|       101|  2|2024-01-02|  200|  Juan|          medio|
|       103|  4|2024-01-03|  300|  Luis|           alto|
+----------+---+----------+-----+------+---------------+



## 12. Cierre breve

**Preguntas para reflexión:**
- ¿Qué diferencia hay entre una transformación simple y un join distribuido?
- ¿Qué aporta la función ventana frente a un `groupBy` tradicional?
- ¿Por qué Parquet es una mejor salida que CSV para analítica posterior?

**Entregable sugerido:** notebook funcional con ETL ejecutado, salida en Parquet y breve interpretación.

# Actividad Autónoma – Aplicación al Proyecto Sello (PS)

## Objetivo
Aplicar el flujo ETL desarrollado en clase sobre el dataset real del proyecto grupal.

## Instrucciones

Cada grupo debe:

1. Leer sus datasets reales del proyecto.
2. Aplicar limpieza de datos:
   - manejo de nulos
   - eliminación de duplicados
3. Integrar al menos dos fuentes mediante join.
4. Aplicar al menos:
   - una función ventana (ranking o acumulado)
5. Validar resultados:
   - conteo de registros
   - verificación de nulos
6. Guardar salida en formato Parquet.
7. Analizar el plan de ejecución (`explain()`).

## Entregable

- Notebook Jupyter (`.ipynb`)
- Código ejecutable
- Resultados visibles
- Breve interpretación (máx. 1 página):
  - qué hicieron
  - qué problemas encontraron
  - qué decisiones tomaron

## Criterios de evaluación

- Correcta implementación del ETL
- Uso adecuado de joins y funciones ventana
- Validación de datos
- Interpretación técnica